# BioHub Cell Tracking — Kaggle Submission Notebook

This notebook's only job: **load the trained model, run the pipeline on the competition's
test data, write `submission.csv` into `/kaggle/working/`.**

All the actual logic (detector, model, tracker, patch classification) still lives in `src/` —
this notebook doesn't reimplement anything, it just calls it in the shape Kaggle expects.

## Before running this on Kaggle

Kaggle competition submissions run with **internet access turned off**, so this notebook
can't `pip install`, `git clone`, or otherwise reach GitHub while it's actually being scored.
That means `src/` and `models/` need to already be sitting inside Kaggle as a **Dataset**,
attached as a notebook input — not fetched at runtime. See `kaggle/README.md` in this repo for
the exact `kaggle datasets create` commands to package and upload them.

Once you have:

1. Created a Kaggle Notebook
2. Added the **competition data** as an input (Add Input → Competitions → this competition)
3. Added your **code dataset** as an input (Add Input → search for the dataset you uploaded —
   see `kaggle/README.md`)
4. Turned **internet OFF** (Notebook settings, right sidebar)

...run this notebook top to bottom. If it finishes without error and `submission.csv` exists
in the output, **Save Version** will make the **Submit** button available.

## ⚠️ One thing you must verify yourself

Cell 9 below (**"Map to the competition's submission format"**) is written from a *reasonable
guess* at column names (`id`, `t`, `z`, `y`, `x`, `parent_id`) based on how tracking-graph
competitions are usually structured. **I could not verify the exact required columns for this
competition** — Kaggle's Data/Evaluation pages are JavaScript-rendered and not something I
could reliably read from outside Kaggle. Cell 9 prints `sample_submission.csv`'s real columns
first specifically so you can check my guess against ground truth before trusting the output.
Please don't skip that check.

## 1. Configuration

Edit `CODE_DATASET_SLUG` to match whatever you name the dataset in `kaggle/README.md`.

In [ ]:
import os
from pathlib import Path

# Change this to "<your-kaggle-username>/<dataset-slug>" once you've uploaded
# src/ + models/ as a Kaggle Dataset (see kaggle/README.md).
CODE_DATASET_SLUG = "your-username/biohub-cell-tracking-src"

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")

print("Available inputs:")
for p in sorted(KAGGLE_INPUT.iterdir()):
    print(" -", p.name)


## 2. Copy code into `/kaggle/working/` and import it

`/kaggle/input/...` is read-only, and code run directly from there can hit import quirks —
copying into the writable working directory first is standard Kaggle practice.

In [ ]:
import shutil
import sys

code_dataset_dir_name = CODE_DATASET_SLUG.split("/")[-1]
code_src = KAGGLE_INPUT / code_dataset_dir_name

if not code_src.exists():
    raise FileNotFoundError(
        f"{code_src} not found. Did you attach the code dataset as an input, and does "
        f"CODE_DATASET_SLUG above match its slug exactly? Available inputs are printed above."
    )

for item in ["src", "models"]:
    dest = KAGGLE_WORKING / item
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(code_src / item, dest)

if str(KAGGLE_WORKING) not in sys.path:
    sys.path.insert(0, str(KAGGLE_WORKING))

from src import config
from src.dataset import BioHubDataset
from src.detector import CellDetector
from src.tracker import HungarianTracker
from src.predict import load_model, classify_centers

print("Imported src OK from", KAGGLE_WORKING)


## 3. Point at the competition's test data

`config.DATASET_PATH` is your local Windows path — irrelevant here. The competition data
Kaggle mounts as a notebook input has its own path under `/kaggle/input/`; this cell finds it
by looking for the folder that isn't the code dataset and contains a `test/` subfolder.

In [ ]:
competition_root = None
for p in sorted(KAGGLE_INPUT.iterdir()):
    if p.name == code_dataset_dir_name:
        continue
    if (p / "test").exists():
        competition_root = p
        break

if competition_root is None:
    raise FileNotFoundError(
        "Couldn't find a competition input folder with a test/ subdirectory. "
        "Did you attach the competition data as an input? Available inputs are printed above — "
        "if the layout doesn't match what this cell expects, set competition_root manually."
    )

print("Competition data root:", competition_root)

dataset = BioHubDataset(competition_root)
print(f"Test samples: {len(dataset.test_samples)}")


## 4. Load the trained model + normalization stats

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = load_model(KAGGLE_WORKING / "models" / "best_model.pth", device)

import numpy as np
stats = np.load(KAGGLE_WORKING / "models" / "norm_stats.npz")
mean, std = float(stats["mean"]), float(stats["std"])

print(f"Model loaded on {device}. mean={mean:.4f}, std={std:.4f}")


## 5. Create detector + tracker

In [ ]:
detector = CellDetector(
    sigma=config.GAUSSIAN_SIGMA,
    threshold_abs=config.DETECTION_THRESHOLD,
    min_distance=config.CELL_RADIUS,
)

tracker = HungarianTracker(max_distance=25)


## 6. Run detection → CNN filtering → tracking, for every test sample

Builds two plain internal tables — `nodes` (one row per accepted cell, every frame, every
sample) and `edges` (one row per frame-to-frame link `HungarianTracker` found) — before
worrying about the competition's exact file format. Keeping this step format-agnostic makes
it easy to re-map into whatever `sample_submission.csv` actually expects (see the next cell)
without having to rerun the whole pipeline.

In [ ]:
nodes = []   # dicts: node_id, sample, t, z, y, x
edges = []   # dicts: sample, source_node_id, target_node_id

node_id_counter = 0

for sample in dataset.test_samples:
    print(f"--- {sample} ---")
    volume = np.asarray(dataset.load_volume(sample, split="test"))

    raw_detections = detector.detect_volume(volume)  # list of (y, x) arrays, one per frame

    # CNN-filter each frame's candidates, and remember each kept detection's
    # global node_id so we can link them across frames below.
    frame_node_ids = []   # frame_node_ids[t] = list of node_id, aligned with frame_coords[t]
    frame_coords = []     # frame_coords[t] = Nx2 array of (y, x), aligned with frame_node_ids[t]

    for t, coords_t in enumerate(raw_detections):
        projection_t = volume[t].max(axis=0)
        kept_t, _ = classify_centers(model, projection_t, coords_t, mean, std, device)

        ids_t = []
        for y, x in kept_t:
            nodes.append({
                "node_id": node_id_counter,
                "sample": sample,
                "t": t,
                "z": 0,  # detection is on the max-projection; see README caveat on 3D z
                "y": int(y),
                "x": int(x),
            })
            ids_t.append(node_id_counter)
            node_id_counter += 1

        frame_node_ids.append(ids_t)
        frame_coords.append(np.array(kept_t) if kept_t else np.empty((0, 2)))

    # Link consecutive frames with the same HungarianTracker used elsewhere in this repo.
    for t in range(len(frame_coords) - 1):
        matches = tracker.match(frame_coords[t], frame_coords[t + 1])
        for m in matches:
            edges.append({
                "sample": sample,
                "source_node_id": frame_node_ids[t][m["previous"]],
                "target_node_id": frame_node_ids[t + 1][m["current"]],
            })

    print(f"  {sum(len(f) for f in frame_node_ids)} nodes, "
          f"{sum(1 for e in edges if e['sample'] == sample)} edges")

print()
print(f"Total: {len(nodes)} nodes, {len(edges)} edges across {len(dataset.test_samples)} samples")


## 7. Check the real submission format before mapping to it

**Run this before trusting the mapping in the next cell.**

In [ ]:
import pandas as pd

sample_submission_path = competition_root / "sample_submission.csv"
if sample_submission_path.exists():
    sample_submission = pd.read_csv(sample_submission_path)
    print("sample_submission.csv columns:", list(sample_submission.columns))
    sample_submission.head()
else:
    print(
        f"No sample_submission.csv found at {sample_submission_path} — "
        "check the competition's Data tab for the exact expected format and "
        "adjust the mapping cell below by hand."
    )


## 8. Map to the competition's submission format

⚠️ **See the warning at the top of this notebook.** This is a best-effort mapping of our
internal `nodes` / `edges` tables into a `(id, t, z, y, x, parent_id)`-style tracking-graph
CSV, which is a common shape for this kind of competition but **not confirmed against this
competition's actual `sample_submission.csv`**. Compare the printed columns above against
what's produced here, and adjust column names / logic to match before submitting.

Nodes with no incoming edge (frame 0, or nothing matched to them) get `parent_id = -1`. Since
this pipeline doesn't yet detect cell divisions (see the repo's Roadmap), no node here ever has
two children — every division-related field / score will legitimately be zero until that's
implemented.

In [ ]:
nodes_df = pd.DataFrame(nodes)
edges_df = pd.DataFrame(edges)

parent_of = {e["target_node_id"]: e["source_node_id"] for e in edges}
nodes_df["parent_id"] = nodes_df["node_id"].map(parent_of).fillna(-1).astype(int)

submission = nodes_df.rename(columns={"node_id": "id"})[["id", "sample", "t", "z", "y", "x", "parent_id"]]

submission.head()


## 9. Save `submission.csv`

In [ ]:
submission_path = KAGGLE_WORKING / "submission.csv"
submission.to_csv(submission_path, index=False)

print(f"Saved {len(submission)} rows to {submission_path}")


## Next steps

1. **Save Version** (top right). Kaggle re-runs this notebook end-to-end on its servers.
2. Wait for it to finish — check the logs if it fails; the most common issue is
   `CODE_DATASET_SLUG` not matching your uploaded dataset's slug, or the competition input
   folder not being where cell 3 expects it.
3. Once it succeeds and `submission.csv` shows up in the version's Output tab, the **Submit**
   button becomes available on that version.
4. Click **Submit** — this is the step that actually turns it in, saving a version alone does
   not submit anything.